# 3.5 读写文件（文件 I/O 与模型参数存取）

[read-write.ipynb](file:///Users/songchao/WorkSpace/Python/DeepLearn/pytorch/chapter_deep-learning-computation/read-write.ipynb) 是《动手学深度学习》中**深度学习计算**章节关于**文件读写与模型序列化（File I/O）**的教学内容。

本节系统讲解了如何持久化存储和读取单个张量、张量集合（列表/字典）以及神经网络模型的参数状态。以下是核心知识点总结：

---

### 一、背景与核心动机

在深度学习的实际工程与科研中，持久化保存数据与模型至关重要：
1. **模型部署与预测（Deployment）**：模型在训练完成后，需要导出保存，以便部署到移动端、服务端或各种生产环境中执行推理预测。
2. **检查点机制（Checkpointing）**：对于耗时数天甚至数周的长时间训练任务，必须定期保存中间状态，以防突发断电、集群故障或异常终止造成计算资源白费。
3. **实验对比与早停选择（Early Stopping）**：在验证集上获得最佳指标时，及时把对应权重存盘，避免后续过拟合阶段覆盖最佳权重。

---

### 二、张量与集合对象的读写（Tensor I/O）

PyTorch 提供了轻量级的底层 API：`torch.save`（序列化到磁盘）与 `torch.load`（从磁盘反序列化）：

#### 1. 单个张量读写
```python
x = torch.arange(4)
torch.save(x, 'x-file')      # 保存为二进制文件
x2 = torch.load('x-file')    # 读取恢复
```

#### 2. 张量列表与元组读写
```python
y = torch.zeros(4)
torch.save([x, y], 'x-files')
x2, y2 = torch.load('x-files')
```

#### 3. 字典映射读写（核心机制）
支持将字符串键映射到张量值的字典对象持久化：
```python
mydict = {'x': x, 'y': y}
torch.save(mydict, 'mydict')
mydict2 = torch.load('mydict')
```
> **说明**：这一功能极为关键，因为神经网络的参数状态字典（`state_dict`）本质上就是由**层名称字符串**映射到**权重/偏置张量**的字典。

---

### 三、模型参数的保存与加载（Model Parameters I/O）

在 PyTorch 等现代深度学习框架中，最推荐的标准做法是**将模型架构（代码定义）与模型参数（权重数值）解耦**。

#### 1. 为什么推荐只保存参数而非整个模型？
* **序列化难度**：模型类定义中往往包含复杂的 Python 动态控制流、外部依赖或自定义代码，直接序列化整个模型对象（如基于 Python `pickle`）容易因代码重命名、类结构微调或跨环境版本差异导致加载失败。
* **解耦优势**：通过固定代码定义架构，仅存储纯数值张量字典，兼容性更好、文件更轻巧，且更易于跨平台迁移。

#### 2. 导出模型参数（`state_dict`）
模型的 `state_dict()` 返回包含了网络所有可学习参数的有序字典（`OrderedDict`）：
```python
# 假设 net 为定义好的多层感知机实例
torch.save(net.state_dict(), 'mlp.params')
```

#### 3. 加载并恢复模型
恢复模型的标准三步：
1. **重新实例化网络结构**；
2. **通过 `torch.load` 读取参数字典**；
3. **调用 `load_state_dict()` 加载权重**。

```python
# 1. 用相同的类代码构建网络架构
clone = MLP()

# 2 & 3. 读取字典并加载进模型
clone.load_state_dict(torch.load('mlp.params'))

# 4. 设为评估模式（冻结 Dropout、BatchNorm 等推理阶段行为）
clone.eval()

# 验证输出一致性
Y_clone = clone(X)
print((Y_clone == net(X)).all())  # 输出 True
```

---

### 四、核心 API 总结表

| 操作需求 | 推荐 API / 方法 | 典型用途与说明 |
| :--- | :--- | :--- |
| **张量序列化** | `torch.save(obj, path)` | 保存单个张量、张量列表或字典至文件 |
| **反序列化读取** | `torch.load(path)` | 从文件加载张量、字典或参数对象到内存 |
| **提取网络参数字典** | `net.state_dict()` | 获取包含模型全部层参数名和张量的字典 |
| **加载权重至网络** | `net.load_state_dict(state_dict)` | 将参数匹配并装载到当前网络各层中 |
| **切换评估模式** | `net.eval()` | 预测前调用，关闭 Dropout、固定 BatchNorm 统计量 |

---

### 五、课后思考题与拓展延伸

1. **除部署外，存储模型参数还有什么实际好处？**
   * **断点续训（Checkpoint）**：遇到中断可以从最后一个检查点继续训练。
   * **最佳模型保留（Early Stopping）**：保存验证集 loss 最低或准确率最高的模型，防止过拟合。
   * **迁移学习与微调（Transfer Learning & Fine-Tuning）**：用大规模数据集（如 ImageNet）预训练好的权重作为下游任务的初始化。
   * **集成学习（Ensembling）**：保存不同训练周期或不同超参数下的多个权重做模型融合。

2. **如何只复用网络的一部分（例如前两层）合并到新网络中？**
   * **字典切片筛选**：加载原模型参数字典后，仅提取前两层键值对，更新到新模型的 `state_dict` 中，或在加载时设置 `strict=False`：
     ```python
     pretrained_dict = torch.load('mlp.params')
     # 过滤出匹配前两层的键
     sub_dict = {k: v for k, v in pretrained_dict.items() if k.startswith('hidden')}
     new_net.load_state_dict(sub_dict, strict=False)
     ```
   * **模块直接挂载**：直接将旧模型的子层赋值给新模型的对应属性（如 `new_net.backbone = pretrained_net.hidden`）。

3. **如何同时保存网络架构与参数？需要对架构加上什么限制？**
   * **方案 A：完整对象保存（`torch.save(net, path)`）**：
     * **限制**：依赖 Python 的 pickle 机制，反序列化时必须能够在相同包路径下找到完全一致的类定义源码，代码重构容易失效。
   * **方案 B：TorchScript（`torch.jit.trace` / `torch.jit.script`）**：
     * 将模型转换为 TorchScript 静态图（`.pt`），包含计算图结构与权重，可脱离 Python 环境（如在 C++ 环境）部署运行。
     * **限制**：代码需要避免动态 Python 语法（如不支持的第三方包、动态类型变化），逻辑必须符合 TorchScript 编译规则。
   * **方案 C：ONNX 导出（`torch.onnx.export`）**：
     * 开放神经网络交换格式，包含网络计算图与张量参数。
     * **限制**：网络操作必须全部属于 ONNX 标准算子集支持范畴。

In [2]:
import torch
from torch.nn import functional as F
from torch import nn

In [3]:
x = torch.arange(4)
torch.save(x, 'x_file')

In [4]:
x2 = torch.load('x_file')

In [5]:
x2

tensor([0, 1, 2, 3])

In [6]:
y = torch.zeros(4)
torch.save([x,y], 'x_files')
x2,y2 = torch.load('x_files')
(x2,y2)

(tensor([0, 1, 2, 3]), tensor([0., 0., 0., 0.]))

In [7]:
mydict = {'x': x, 'y': y}
torch.save(mydict, 'mydict')
mydict2 = torch.load('mydict')
mydict2

{'x': tensor([0, 1, 2, 3]), 'y': tensor([0., 0., 0., 0.])}

In [8]:
class MLP(nn.Module):

    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(20, 256)
        self.output = nn.Linear(256, 10)
    
    def forward(self, X):
        return self.output(F.relu((self.hidden(X))))


In [9]:
net = MLP()
X = torch.randn(size=(2,20))
Y = net(X)

In [11]:
torch.save(net.state_dict(), 'mlp.params')

In [12]:
clone = MLP()
clone.load_state_dict(torch.load('mlp.params'))
clone.eval()

MLP(
  (hidden): Linear(in_features=20, out_features=256, bias=True)
  (output): Linear(in_features=256, out_features=10, bias=True)
)

In [13]:
Y_clone = clone(X)
Y_clone == Y

tensor([[True, True, True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True]])